# Spotify Analysis - Parte 4

Los pasos están organizados en secciones:

1. Importar librerías y cargar datos
2. Fechas de lanzamiento y medias por fecha
3. Definir eras y agrupar características
4. Barras de promedios por era
5. Agregar popularidad/streams por mes para canciones top

> **Nota:** no existe una columna explícita de "streams" en la base de datos; sólo se dispone de `track_popularity`. En consecuencia, la agregación mensual usará popularidad como aproximación.

In [ ]:
# Section 1: Import Libraries and Load Data
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# connect to sqlite and load tables
conn = sqlite3.connect(r'c:\Users\smart\Downloads\Spotifiy_4-Mario\Spotifiy_4\spotify_database.db')

albums = pd.read_sql_query('SELECT * FROM albums_data', conn)
features = pd.read_sql_query('SELECT * FROM features_data', conn)
tracks = pd.read_sql_query('SELECT * FROM tracks_data', conn)

# close connection when done (data now in memory)
conn.close()

# quick look
print(albums.head())
print(features.head())
print(tracks.head())

In [ ]:
# Parse Release Dates and Compute Averages Over Time
# convert release_date to datetime (year-month-day, some may be year only)
albums['release_date'] = pd.to_datetime(albums['release_date'], errors='coerce')

# join audio features with release date via track id / id
# features.id corresponds to track_id in albums
merged = albums.merge(features, left_on='track_id', right_on='id', how='inner')

# compute average of audio features for each release_date
audio_cols = ['danceability','energy','loudness','speechiness','acousticness',
              'instrumentalness','liveness','valence','tempo']
daily_avg = merged.groupby('release_date')[audio_cols].mean().reset_index()

# show sample
daily_avg.head()

In [ ]:
# Define Eras and Group Features
# determine release year and categorize
merged['year'] = merged['release_date'].dt.year

def categorize_era(year):
    if pd.isna(year):
        return 'unknown'
    if year < 1980:
        return 'pre-1980'
    elif year < 2000:
        return '1980-1999'
    elif year < 2010:
        return '2000-2009'
    elif year < 2020:
        return '2010-2019'
    else:
        return '2020+'

merged['era'] = merged['year'].apply(categorize_era)

# aggregate features by era
era_avg = merged.groupby('era')[audio_cols].mean().reset_index()
era_avg

In [ ]:
# Create Barplots for Feature Averages by Era
sns.set(style='whitegrid')
plt.figure(figsize=(10,6))
for feature in ['danceability','energy','valence']:
    sns.barplot(data=era_avg, x='era', y=feature)
    plt.title(f'Average {feature.capitalize()} by Era')
    plt.ylabel(feature.capitalize())
    plt.xlabel('Era')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# Section 5: Aggregate Popularity by Month for Top-Streamed (Popular) Songs
# join popularity from tracks table
pop_df = merged.merge(tracks, left_on='track_id', right_on='id', how='left')

# take top 100 by popularity
top = pop_df.nlargest(100, 'track_popularity').copy()

# create month column from release_date
top['month'] = top['release_date'].dt.to_period('M')
monthly = top.groupby('month')[['track_popularity']].mean().reset_index()

# plot
plt.figure(figsize=(12,4))
sns.lineplot(data=monthly, x='month', y='track_popularity', marker='o')
plt.xticks(rotation=45)
plt.title('Average Popularity by Month for Top 100 Tracks')
plt.xlabel('Month')
plt.ylabel('Popularity')
plt.tight_layout()
plt.show()